### Classification

**Requirements (advanced dataset)**
1. Shape into classification with a fixed number of classes (3: low/medium/high)
2. Two different *types* of algorithm:
   - **Model A**: non-temporal, uses the 1C flat dataset → XGBoost
   - **Model B**: inherently temporal → LSTM (PyTorch)
3. Time-aware evaluation setup
4. Hyperparameter optimization
5. Performance measurement with justified metric

#### 1. Configuration & Loading

In [3]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("../outputs")
FIG_DIR    = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load the 1C modelling dataset ─────────────────────────────────────
df = pd.read_csv(OUTPUT_DIR / "data_modelling.csv", parse_dates=["date", "target_date"])
df = df.sort_values(["date", "id"]).reset_index(drop=True)

print(f"Loaded {len(df)} instances, {df['id'].nunique()} patients")
print(f"Date range: {df['date'].min().date()} – {df['date'].max().date()}")
print(f"Target range: {df['target_next_day_mood'].min():.2f} – "
      f"{df['target_next_day_mood'].max():.2f}")

Loaded 1394 instances, 27 patients
Date range: 2014-01-03 – 2014-12-04
Target range: 3.00 – 9.33


#### 2. Target Discretization

We bin next-day mood into **3 classes** using **tertile boundaries** computed on
the full dataset.  Tertiles produce roughly balanced classes, which is important
because (a) macro F1 is our metric and it penalises poor performance on any
single class, and (b) the left-skewed mood distribution (mean 7.0, range 3–9.3)
would create severe imbalance under fixed thresholds.

The thresholds are fixed globally rather than per-fold to ensure consistent
class definitions across all CV folds.

In [4]:
# Compute tertile boundaries
t33 = df["target_next_day_mood"].quantile(0.333)
t67 = df["target_next_day_mood"].quantile(0.667)

def discretize_mood(values, low_hi=t33, med_hi=t67):
    """Bin continuous mood into 3 classes."""
    return pd.cut(
        values,
        bins=[-np.inf, low_hi, med_hi, np.inf],
        labels=["low", "medium", "high"]
    ).astype(str)

df["mood_class"] = discretize_mood(df["target_next_day_mood"])

# Encode as integers for models
CLASS_MAP = {"low": 0, "medium": 1, "high": 2}
CLASS_NAMES = ["low", "medium", "high"]
df["mood_class_int"] = df["mood_class"].map(CLASS_MAP)

print(f"Tertile boundaries: low ≤ {t33:.3f} < medium ≤ {t67:.3f} < high")
print(f"\nClass distribution:")
for cls, count in df["mood_class"].value_counts().sort_index().items():
    print(f"  {cls:8s}: {count:5d}  ({count/len(df)*100:.1f}%)")

Tertile boundaries: low ≤ 6.800 < medium ≤ 7.250 < high

Class distribution:
  high    :   447  (32.1%)
  low     :   508  (36.4%)
  medium  :   439  (31.5%)


#### 3. Walk-Forward Temporal CV

**Why time-aware CV is essential**: the data is a collection of time series.
A random train/test split would allow the model to train on future observations
and predict past ones — temporal leakage.  Walk-forward (expanding window) CV
respects the arrow of time: each fold trains on all data before a date cutoff
and tests on the next temporal block.

We use 4 folds, each with a progressively larger training set and a fixed-size
test block (~20% of the date range each).

In [5]:
# Create temporal folds based on date quantiles
unique_dates = sorted(df["date"].unique())
n_dates = len(unique_dates)

# Split date range into 5 equal blocks → 4 test folds
block_size = n_dates // 5
date_blocks = []
for i in range(5):
    start = i * block_size
    end = (i + 1) * block_size if i < 4 else n_dates
    date_blocks.append(set(unique_dates[start:end]))

# Walk-forward folds: train on blocks [0..k], test on block [k+1]
cv_folds = []
for k in range(4):
    train_dates = set()
    for j in range(k + 1):
        train_dates |= date_blocks[j]
    test_dates = date_blocks[k + 1]

    train_idx = df[df["date"].isin(train_dates)].index.tolist()
    test_idx  = df[df["date"].isin(test_dates)].index.tolist()
    cv_folds.append((train_idx, test_idx))

print("Walk-forward CV folds:")
for i, (tr, te) in enumerate(cv_folds):
    tr_dates = df.loc[tr, "date"]
    te_dates = df.loc[te, "date"]
    print(f"  Fold {i+1}: train {len(tr):4d} instances "
          f"({tr_dates.min().date()}–{tr_dates.max().date()}) │ "
          f"test {len(te):4d} instances "
          f"({te_dates.min().date()}–{te_dates.max().date()})")

Walk-forward CV folds:
  Fold 1: train  203 instances (2014-01-03–2014-03-20) │ test  395 instances (2014-03-21–2014-04-15)
  Fold 2: train  598 instances (2014-01-03–2014-04-15) │ test  490 instances (2014-04-16–2014-05-05)
  Fold 3: train 1088 instances (2014-01-03–2014-05-05) │ test   67 instances (2014-05-12–2014-06-02)
  Fold 4: train 1155 instances (2014-01-03–2014-06-02) │ test  239 instances (2014-06-03–2014-12-04)


#### 4. Evaluation Helpers

In [6]:
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, accuracy_score
)

def evaluate_fold(y_true, y_pred, fold_name=""):
    """Compute metrics for one fold. Returns dict."""
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return {
        "fold": fold_name,
        "macro_f1": macro_f1,
        "accuracy": acc,
        "n_test": len(y_true),
    }

def print_summary(results, model_name):
    """Print CV summary from list of fold result dicts."""
    f1s = [r["macro_f1"] for r in results]
    accs = [r["accuracy"] for r in results]
    print(f"\n{'='*60}")
    print(f"{model_name} — CV Summary")
    print(f"{'='*60}")
    for r in results:
        print(f"  {r['fold']}: macro-F1 = {r['macro_f1']:.4f}  "
              f"acc = {r['accuracy']:.4f}  (n={r['n_test']})")
    print(f"  {'─'*50}")
    print(f"  Mean macro-F1 = {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"  Mean accuracy = {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return np.mean(f1s), np.std(f1s)

#### 5. Baseline — Majority Class

Before running any model, we establish the floor: predicting the most frequent
class every time.

In [7]:
baseline_results = []
for i, (train_idx, test_idx) in enumerate(cv_folds):
    y_train = df.loc[train_idx, "mood_class_int"].values
    y_test  = df.loc[test_idx, "mood_class_int"].values
    majority = Counter(y_train).most_common(1)[0][0]
    y_pred = np.full_like(y_test, majority)
    baseline_results.append(evaluate_fold(y_test, y_pred, f"Fold {i+1}"))

baseline_f1, baseline_std = print_summary(baseline_results, "Majority-Class Baseline")


Majority-Class Baseline — CV Summary
  Fold 1: macro-F1 = 0.1772  acc = 0.3620  (n=395)
  Fold 2: macro-F1 = 0.1656  acc = 0.3306  (n=490)
  Fold 3: macro-F1 = 0.2062  acc = 0.4478  (n=67)
  Fold 4: macro-F1 = 0.1809  acc = 0.3724  (n=239)
  ──────────────────────────────────────────────────
  Mean macro-F1 = 0.1825 ± 0.0148
  Mean accuracy = 0.3782 ± 0.0430


#### 6. Model A — XGBoost on Flat Windowed Features (windowed feature dataset)

**Why XGBoost:**
- Handles NaN natively (can use all 1,394 instances including Tier B/C)
- Strong gradient-boosted ensemble, widely used in practice
- Provides feature importance for interpretability
- Efficient hyperparameter search

**Hyperparameter optimization:** RandomizedSearchCV over the temporal CV folds.

In [8]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# Feature columns (everything except meta + target + quality_tier)
meta_cols = ["id", "date", "target_date", "target_next_day_mood",
             "mood_class", "mood_class_int", "quality_tier"]
feat_cols = [c for c in df.columns if c not in meta_cols]

X = df[feat_cols].values
y = df["mood_class_int"].values

print(f"Feature matrix: {X.shape}")
print(f"Features: {feat_cols}")

Feature matrix: (1394, 23)
Features: ['activity_lag1', 'activity_w5_mean', 'activity_w5_std', 'activity_w5_trend', 'appCat_communication_lag1', 'appCat_communication_w5_mean', 'appCat_communication_w5_std', 'appCat_communication_w5_trend', 'circumplex_valence_lag1', 'circumplex_valence_w5_mean', 'circumplex_valence_w5_std', 'circumplex_valence_w5_trend', 'dow_cos', 'dow_sin', 'mood_lag1', 'mood_lag2', 'mood_w5_mean', 'mood_w5_std', 'mood_w5_trend', 'screen_lag1', 'screen_w5_mean', 'screen_w5_std', 'screen_w5_trend']


##### 6a. Hyperparameter Search

In [9]:
param_dist = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 4, 5, 6, 8],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight":  [1, 3, 5, 7],
    "gamma":             [0, 0.1, 0.2, 0.5],
    "reg_alpha":         [0, 0.01, 0.1, 1.0],
    "reg_lambda":        [0.5, 1.0, 2.0, 5.0],
}

xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    verbosity=0,
    enable_categorical=False,
)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring="f1_macro",
    cv=cv_folds,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=False,         # we refit manually per fold for proper reporting
)

search.fit(X, y)
best_params = search.best_params_
print(f"\nBest hyperparameters (macro-F1 = {search.best_score_:.4f}):")
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

Fitting 4 folds for each of 60 candidates, totalling 240 fits

Best hyperparameters (macro-F1 = 0.4441):
  colsample_bytree: 0.7
  gamma: 0.2
  learning_rate: 0.05
  max_depth: 6
  min_child_weight: 7
  n_estimators: 200
  reg_alpha: 0.01
  reg_lambda: 5.0
  subsample: 0.7


##### 6b. Per-Fold Evaluation with Best Params

In [10]:
xgb_results = []
xgb_all_y_true = []
xgb_all_y_pred = []

for i, (train_idx, test_idx) in enumerate(cv_folds):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = XGBClassifier(
        **best_params,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42,
        verbosity=0,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    xgb_results.append(evaluate_fold(y_test, y_pred, f"Fold {i+1}"))
    xgb_all_y_true.extend(y_test)
    xgb_all_y_pred.extend(y_pred)

xgb_f1, xgb_std = print_summary(xgb_results, "XGBoost")

# Pooled classification report
print(f"\nPooled Classification Report (XGBoost):")
print(classification_report(
    xgb_all_y_true, xgb_all_y_pred,
    target_names=CLASS_NAMES, digits=3, zero_division=0
))

# Pooled confusion matrix
cm_xgb = confusion_matrix(xgb_all_y_true, xgb_all_y_pred)
print("Confusion Matrix (rows=true, cols=predicted):")
print(pd.DataFrame(cm_xgb, index=CLASS_NAMES, columns=CLASS_NAMES))


XGBoost — CV Summary
  Fold 1: macro-F1 = 0.4120  acc = 0.4380  (n=395)
  Fold 2: macro-F1 = 0.5746  acc = 0.5735  (n=490)
  Fold 3: macro-F1 = 0.4025  acc = 0.4776  (n=67)
  Fold 4: macro-F1 = 0.3672  acc = 0.3682  (n=239)
  ──────────────────────────────────────────────────
  Mean macro-F1 = 0.4391 ± 0.0800
  Mean accuracy = 0.4643 ± 0.0742

Pooled Classification Report (XGBoost):
              precision    recall  f1-score   support

         low      0.499     0.592     0.542       424
      medium      0.424     0.420     0.422       383
        high      0.526     0.422     0.468       384

    accuracy                          0.482      1191
   macro avg      0.483     0.478     0.477      1191
weighted avg      0.483     0.482     0.479      1191

Confusion Matrix (rows=true, cols=predicted):
        low  medium  high
low     251     109    64
medium  140     161    82
high    112     110   162


##### 6c. Feature Importance

In [11]:
# Retrain on full data for feature importance (for reporting only)
final_xgb = XGBClassifier(
    **best_params,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    verbosity=0,
)
final_xgb.fit(X, y)

importances = pd.Series(final_xgb.feature_importances_, index=feat_cols)
importances = importances.sort_values(ascending=False)

print("Feature importance (top 15):")
for fname, imp in importances.head(15).items():
    print(f"  {fname:40s}  {imp:.4f}")

# Save for report
importances.to_csv(OUTPUT_DIR / "xgb_feature_importance.csv")

Feature importance (top 15):
  mood_lag1                                 0.1050
  mood_w5_mean                              0.0716
  mood_lag2                                 0.0485
  activity_w5_std                           0.0479
  circumplex_valence_w5_mean                0.0462
  mood_w5_std                               0.0424
  appCat_communication_w5_mean              0.0415
  circumplex_valence_w5_trend               0.0402
  screen_w5_mean                            0.0401
  activity_w5_trend                         0.0395
  screen_lag1                               0.0383
  activity_w5_mean                          0.0382
  activity_lag1                             0.0382
  appCat_communication_lag1                 0.0381
  circumplex_valence_lag1                   0.0379


#### 7. Model B — LSTM on Raw Daily Sequences (Inherently Temporal)

**Why LSTM:**
- Learns temporal patterns directly from the sequence of daily observations
- Does not require hand-crafted lag/rolling features — the network learns
  its own temporal aggregation
- Fundamentally different approach from XGBoost, satisfying the "two types"
  requirement

**Data pipeline:** for each modelling instance, extract the W=5 preceding days
of raw feature values from `data_imputed.csv`, forming a (5 × 5) sequence
matrix.  Only fully complete sequences are used (no NaN in any timestep).

**Architecture:** 1-layer LSTM → dropout → linear → 3-class softmax.

##### 7a. Sequence Preparation

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load the imputed daily panel
df_daily = pd.read_csv(OUTPUT_DIR / "data_imputed.csv", parse_dates=["date"])
df_daily = df_daily.sort_values(["id", "date"]).reset_index(drop=True)

# Apply the same sqrt transform to activity (only transform that passed the
# adaptive check in 1C)
df_daily["activity_t"] = np.sqrt(df_daily["activity"])

# Features to extract per timestep
SEQ_FEATURES = ["mood", "circumplex.valence", "activity_t", "screen",
                "appCat.communication"]

W = 5  # same window as 1C

# Build a lookup: (id, date) → feature vector
df_daily_indexed = df_daily.set_index(["id", "date"])[SEQ_FEATURES]

# For each modelling instance, extract the W-day sequence
sequences = []
targets = []
instance_dates = []     # for temporal CV alignment
instance_indices = []   # index into df for matching

for idx, row in df.iterrows():
    pid = row["id"]
    end_date = row["date"]  # last day of feature window
    target_cls = row["mood_class_int"]

    # Build sequence: [end_date - W + 1, ..., end_date]
    seq_dates = pd.date_range(end=end_date, periods=W, freq="D")

    try:
        seq = df_daily_indexed.loc[
            [(pid, d) for d in seq_dates]
        ].values  # shape (W, n_features)
    except KeyError:
        continue  # some dates may be missing from the panel

    # Skip sequences with any NaN
    if np.isnan(seq).any():
        continue

    sequences.append(seq)
    targets.append(target_cls)
    instance_dates.append(end_date)
    instance_indices.append(idx)

sequences = np.array(sequences, dtype=np.float32)   # (N, W, F)
targets = np.array(targets, dtype=np.int64)
instance_dates = np.array(instance_dates)
instance_indices = np.array(instance_indices)

print(f"Complete sequences: {len(sequences)} / {len(df)} instances "
      f"({len(sequences)/len(df)*100:.1f}%)")
print(f"Sequence shape: {sequences.shape}")
print(f"Class distribution in sequences:")
for cls_name, cls_int in CLASS_MAP.items():
    n = (targets == cls_int).sum()
    print(f"  {cls_name}: {n}  ({n/len(targets)*100:.1f}%)")

Complete sequences: 660 / 1394 instances (47.3%)
Sequence shape: (660, 5, 5)
Class distribution in sequences:
  low: 221  (33.5%)
  medium: 217  (32.9%)
  high: 222  (33.6%)


##### 7b. Temporal CV Folds for LSTM

The complete sequences are concentrated in a narrower date range than the
full dataset (the early months have too much missingness for complete 5-day
windows, and the late months have few remaining patients).  We therefore
build the LSTM CV folds from the **sequence dates themselves** using the same
walk-forward expanding-window logic, ensuring every fold has both training
and test data.  This is methodologically equivalent to the XGBoost CV — both
use temporal walk-forward — but adapted to the LSTM's available data range.

In [13]:
# Build LSTM-specific temporal folds from the sequence dates
seq_dates_unique = sorted(set(instance_dates))
n_seq_dates = len(seq_dates_unique)

# Split the sequence date range into 4 equal blocks → 3 walk-forward folds
lstm_block_size = n_seq_dates // 4
lstm_date_blocks = []
for i in range(4):
    start = i * lstm_block_size
    end = (i + 1) * lstm_block_size if i < 3 else n_seq_dates
    lstm_date_blocks.append(set(seq_dates_unique[start:end]))

lstm_cv_folds = []
for k in range(3):  # 3 folds
    train_dates = set()
    for j in range(k + 1):
        train_dates |= lstm_date_blocks[j]
    test_dates = lstm_date_blocks[k + 1]

    train_seq_idx = [i for i, d in enumerate(instance_dates) if d in train_dates]
    test_seq_idx  = [i for i, d in enumerate(instance_dates) if d in test_dates]

    lstm_cv_folds.append((train_seq_idx, test_seq_idx))

print("LSTM walk-forward CV folds (built from sequence dates):")
for fi, (tr, te) in enumerate(lstm_cv_folds):
    tr_d = instance_dates[tr] if len(tr) > 0 else []
    te_d = instance_dates[te] if len(te) > 0 else []
    tr_min = pd.Timestamp(min(tr_d)).strftime('%Y-%m-%d') if len(tr_d) else 'N/A'
    tr_max = pd.Timestamp(max(tr_d)).strftime('%Y-%m-%d') if len(tr_d) else 'N/A'
    te_min = pd.Timestamp(min(te_d)).strftime('%Y-%m-%d') if len(te_d) else 'N/A'
    te_max = pd.Timestamp(max(te_d)).strftime('%Y-%m-%d') if len(te_d) else 'N/A'
    print(f"  Fold {fi+1}: train {len(tr):4d}  "
          f"({tr_min}–{tr_max}) │ "
          f"test {len(te):4d}  "
          f"({te_min}–{te_max})")

LSTM walk-forward CV folds (built from sequence dates):
  Fold 1: train  130  (2014-03-21–2014-03-31) │ test  243  (2014-04-01–2014-04-23)
  Fold 2: train  373  (2014-03-21–2014-04-23) │ test  246  (2014-04-24–2014-05-04)
  Fold 3: train  619  (2014-03-21–2014-05-04) │ test   41  (2014-05-17–2014-05-30)


##### 7c. LSTM Architecture

In [14]:
class MoodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes=3,
                 num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, _) = self.lstm(x)
        # Use final hidden state
        out = self.dropout(h_n[-1])     # (batch, hidden_size)
        out = self.fc(out)              # (batch, num_classes)
        return out


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

##### 7d. LSTM Training & Hyperparameter Search

We search over a small grid of configurations (hidden size, learning rate,
dropout) using the walk-forward CV folds.  For each configuration, we compute
the mean macro-F1 across folds and select the best.

In [15]:
from sklearn.preprocessing import StandardScaler

def train_lstm_fold(X_train, y_train, X_test, y_test,
                    hidden_size=64, lr=0.001, dropout=0.3,
                    epochs=80, batch_size=32, patience=15):
    """Train LSTM on one fold, return test predictions."""

    # Standardize features (fit on train only)
    n_train, seq_len, n_feat = X_train.shape
    scaler = StandardScaler()
    X_tr_flat = X_train.reshape(-1, n_feat)
    scaler.fit(X_tr_flat)
    X_train_s = scaler.transform(X_tr_flat).reshape(n_train, seq_len, n_feat)
    X_test_s  = scaler.transform(
        X_test.reshape(-1, n_feat)
    ).reshape(X_test.shape[0], seq_len, n_feat)

    # Datasets
    train_ds = SequenceDataset(X_train_s, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MoodLSTM(
        input_size=n_feat, hidden_size=hidden_size,
        num_classes=3, dropout=dropout
    ).to(device)

    # Class weights for imbalanced classes
    class_counts = np.bincount(y_train, minlength=3).astype(float)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * 3
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )

    # Training loop with early stopping
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(y_batch)

        epoch_loss /= len(train_ds)
        scheduler.step(epoch_loss)

        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Load best model and predict
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_test_s, dtype=torch.float32).to(device)
        logits = model(X_t)
        y_pred = logits.argmax(dim=1).cpu().numpy()

    return y_pred, epoch + 1


# Hyperparameter grid
lstm_configs = [
    {"hidden_size": 32,  "lr": 0.001, "dropout": 0.2},
    {"hidden_size": 32,  "lr": 0.005, "dropout": 0.3},
    {"hidden_size": 64,  "lr": 0.001, "dropout": 0.3},
    {"hidden_size": 64,  "lr": 0.005, "dropout": 0.2},
    {"hidden_size": 64,  "lr": 0.01,  "dropout": 0.3},
    {"hidden_size": 128, "lr": 0.001, "dropout": 0.3},
]

print(f"LSTM hyperparameter search: {len(lstm_configs)} configurations "
      f"× {len(lstm_cv_folds)} folds")
print()

best_lstm_f1 = -1
best_lstm_config = None
config_results = []

for ci, config in enumerate(lstm_configs):
    fold_f1s = []
    for fi, (tr_idx, te_idx) in enumerate(lstm_cv_folds):
        if len(tr_idx) == 0 or len(te_idx) == 0:
            continue
        X_tr = sequences[tr_idx]
        y_tr = targets[tr_idx]
        X_te = sequences[te_idx]
        y_te = targets[te_idx]

        y_pred, n_epochs = train_lstm_fold(
            X_tr, y_tr, X_te, y_te, **config
        )
        f1 = f1_score(y_te, y_pred, average="macro", zero_division=0)
        fold_f1s.append(f1)

    mean_f1 = np.mean(fold_f1s) if fold_f1s else 0
    config_results.append({"config": config, "mean_f1": mean_f1,
                           "fold_f1s": fold_f1s})
    print(f"  Config {ci+1}: h={config['hidden_size']:3d}  "
          f"lr={config['lr']:.3f}  drop={config['dropout']:.1f}  "
          f"→ mean F1 = {mean_f1:.4f}")

    if mean_f1 > best_lstm_f1:
        best_lstm_f1 = mean_f1
        best_lstm_config = config

print(f"\nBest LSTM config: {best_lstm_config}  (F1 = {best_lstm_f1:.4f})")

LSTM hyperparameter search: 6 configurations × 3 folds

  Config 1: h= 32  lr=0.001  drop=0.2  → mean F1 = 0.4506
  Config 2: h= 32  lr=0.005  drop=0.3  → mean F1 = 0.4421
  Config 3: h= 64  lr=0.001  drop=0.3  → mean F1 = 0.4318
  Config 4: h= 64  lr=0.005  drop=0.2  → mean F1 = 0.4434
  Config 5: h= 64  lr=0.010  drop=0.3  → mean F1 = 0.4298
  Config 6: h=128  lr=0.001  drop=0.3  → mean F1 = 0.4216

Best LSTM config: {'hidden_size': 32, 'lr': 0.001, 'dropout': 0.2}  (F1 = 0.4506)


##### 7e. Final LSTM Evaluation with Best Config

In [16]:
torch.manual_seed(42)
np.random.seed(42)

lstm_results = []
lstm_all_y_true = []
lstm_all_y_pred = []

for i, (tr_idx, te_idx) in enumerate(lstm_cv_folds):
    if len(tr_idx) == 0 or len(te_idx) == 0:
        print(f"  Fold {i+1}: SKIPPED (train={len(tr_idx)}, test={len(te_idx)})")
        continue
    X_tr = sequences[tr_idx]
    y_tr = targets[tr_idx]
    X_te = sequences[te_idx]
    y_te = targets[te_idx]

    y_pred, n_epochs = train_lstm_fold(
        X_tr, y_tr, X_te, y_te, **best_lstm_config
    )

    lstm_results.append(evaluate_fold(y_te, y_pred, f"Fold {i+1}"))
    lstm_all_y_true.extend(y_te)
    lstm_all_y_pred.extend(y_pred)
    print(f"  Fold {i+1}: F1={lstm_results[-1]['macro_f1']:.4f}  "
          f"(trained {n_epochs} epochs, n_test={len(y_te)})")

lstm_f1, lstm_std = print_summary(lstm_results, "LSTM")

print(f"\nPooled Classification Report (LSTM):")
print(classification_report(
    lstm_all_y_true, lstm_all_y_pred,
    target_names=CLASS_NAMES, digits=3, zero_division=0
))

cm_lstm = confusion_matrix(lstm_all_y_true, lstm_all_y_pred)
print("Confusion Matrix (rows=true, cols=predicted):")
print(pd.DataFrame(cm_lstm, index=CLASS_NAMES, columns=CLASS_NAMES))

  Fold 1: F1=0.4912  (trained 80 epochs, n_test=243)
  Fold 2: F1=0.5316  (trained 80 epochs, n_test=246)
  Fold 3: F1=0.2566  (trained 80 epochs, n_test=41)

LSTM — CV Summary
  Fold 1: macro-F1 = 0.4912  acc = 0.4897  (n=243)
  Fold 2: macro-F1 = 0.5316  acc = 0.5285  (n=246)
  Fold 3: macro-F1 = 0.2566  acc = 0.3902  (n=41)
  ──────────────────────────────────────────────────
  Mean macro-F1 = 0.4265 ± 0.1212
  Mean accuracy = 0.4695 ± 0.0582

Pooled Classification Report (LSTM):
              precision    recall  f1-score   support

         low      0.591     0.523     0.555       174
      medium      0.414     0.480     0.444       171
        high      0.517     0.497     0.507       185

    accuracy                          0.500       530
   macro avg      0.507     0.500     0.502       530
weighted avg      0.508     0.500     0.502       530

Confusion Matrix (rows=true, cols=predicted):
        low  medium  high
low      91      53    30
medium   33      82    56
high   

#### 8. Model Comparison & Interpretation

In [17]:
print(f"\n{'='*60}")
print("Model Comparison")
print(f"{'='*60}")
print(f"  {'Model':<25s}  {'Macro-F1':>10s}  {'± Std':>8s}  {'Instances':>10s}")
print(f"  {'─'*55}")
print(f"  {'Majority baseline':<25s}  {baseline_f1:>10.4f}  {baseline_std:>8.4f}  "
      f"{'1394':>10s}")
print(f"  {'XGBoost (1C features)':<25s}  {xgb_f1:>10.4f}  {xgb_std:>8.4f}  "
      f"{'1394':>10s}")
print(f"  {'LSTM (sequences)':<25s}  {lstm_f1:>10.4f}  {lstm_std:>8.4f}  "
      f"{str(len(sequences)):>10s}")

# Improvement over baseline
xgb_lift = xgb_f1 - baseline_f1
lstm_lift = lstm_f1 - baseline_f1
print(f"\n  XGBoost lift over baseline: {xgb_lift:+.4f}")
print(f"  LSTM lift over baseline:    {lstm_lift:+.4f}")

if xgb_f1 > lstm_f1:
    better = "XGBoost"
    delta = xgb_f1 - lstm_f1
else:
    better = "LSTM"
    delta = lstm_f1 - xgb_f1
print(f"\n  → {better} outperforms by {delta:.4f} macro-F1")


Model Comparison
  Model                        Macro-F1     ± Std   Instances
  ───────────────────────────────────────────────────────
  Majority baseline              0.1825    0.0148        1394
  XGBoost (1C features)          0.4391    0.0800        1394
  LSTM (sequences)               0.4265    0.1212         660

  XGBoost lift over baseline: +0.2566
  LSTM lift over baseline:    +0.2440

  → XGBoost outperforms by 0.0126 macro-F1


#### 9. Export Results

In [18]:
# Save comparison table
comparison = pd.DataFrame([
    {"model": "majority_baseline", "macro_f1_mean": baseline_f1,
     "macro_f1_std": baseline_std, "n_instances": 1394},
    {"model": "xgboost", "macro_f1_mean": xgb_f1,
     "macro_f1_std": xgb_std, "n_instances": 1394,
     "best_params": str(best_params)},
    {"model": "lstm", "macro_f1_mean": lstm_f1,
     "macro_f1_std": lstm_std, "n_instances": len(sequences),
     "best_params": str(best_lstm_config)},
])
comparison.to_csv(OUTPUT_DIR / "classification_results.csv", index=False)

# Save confusion matrices
pd.DataFrame(cm_xgb, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    OUTPUT_DIR / "confusion_matrix_xgboost.csv")
pd.DataFrame(cm_lstm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    OUTPUT_DIR / "confusion_matrix_lstm.csv")

# Save XGBoost best params
pd.Series(best_params).to_csv(OUTPUT_DIR / "xgb_best_params.csv")

print(f"\nSaved to {OUTPUT_DIR}:")
print(f"  classification_results.csv")
print(f"  confusion_matrix_xgboost.csv")
print(f"  confusion_matrix_lstm.csv")
print(f"  xgb_feature_importance.csv")
print(f"  xgb_best_params.csv")

print(f"\n✅ Task 2A complete.")


Saved to ..\outputs:
  classification_results.csv
  confusion_matrix_xgboost.csv
  confusion_matrix_lstm.csv
  xgb_feature_importance.csv
  xgb_best_params.csv

✅ Task 2A complete.


---

**Metric choice: macro F1-score.**
We use macro F1 because it gives equal weight to all three mood classes
regardless of their frequency. Accuracy alone would be misleading if the
classifier favours the majority class. Macro F1 penalises models that
sacrifice performance on any single class, which is important since
predicting low mood (the clinically most relevant class) is as important
as predicting medium or high.

**Evaluation setup: walk-forward expanding-window CV.**
This design respects the temporal ordering of the data. Each fold trains
only on past observations and evaluates on future ones, mimicking the real
deployment scenario: given a patient's history, predict tomorrow's mood.
A random train/test split would allow temporal leakage and inflate
performance estimates.

**XGBoost advantages:**
- Can use all 1,394 instances (including those with partial feature coverage)
  because it handles missing values natively
- Feature importance reveals which temporal aggregates matter most
- Fast to train and tune

**LSTM advantages:**
- Learns temporal patterns directly from raw sequences without hand-crafted
  features — the network discovers its own aggregation strategy
- Theoretically more expressive for complex temporal dependencies

**LSTM limitations in this setting:**
- Restricted to complete sequences only (~660 of 1,394 instances)
- Small dataset for deep learning (660 sequences of length 5)
- Less interpretable than XGBoost

If XGBoost outperforms LSTM, this suggests that for small EMA datasets,
carefully engineered temporal features (lag, rolling mean, trend) capture
the relevant signal more effectively than end-to-end sequence learning.
The hand-crafted features embed domain knowledge (e.g., which variables
matter, what window size to use) that the LSTM must learn from limited data.
This is a common finding in applied mental-health prediction and worth
stating explicitly.